<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week7_Day7_Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [20]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [21]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [22]:
from datasets import load_dataset

# Nom du jeu de données sur le Hub Hugging Face
dataset_name = "m-ric/huggingface_doc"

# Chargement des 200 premières lignes du split 'train'
# Cela permet de garder l'exercice rapide et fluide
ds = load_dataset(dataset_name, split="train[:200]")

# Affichage des informations de base pour vérifier le chargement
print(f"Jeu de données chargé avec succès !")
print(f"Nombre de lignes : {len(ds)}")
print(f"Colonnes disponibles : {ds.column_names}")

# Aperçu de la première ligne pour voir la structure des données
display(ds[0])

Jeu de données chargé avec succès !
Nombre de lignes : 200
Colonnes disponibles : ['text', 'source']


{'text': ' Create an Endpoint\n\nAfter your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. \n\n## 1. Enter the Hugging Face Repository ID and your desired endpoint name:\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />\n\n## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.\n\n<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_region.png" alt="select region" />\n\n## 3. Defi

In [23]:
from langchain_core.documents import Document

# Nous allons extraire le texte et la source de chaque ligne du dataset
text_column = "text"
source_column = "source"

documents = []
for row in ds:
    # Création d'un objet Document
    doc = Document(
        page_content=row[text_column], # Le texte que l'IA va lire
        metadata={"source": row[source_column]} # L'origine du texte pour le traçage
    )
    documents.append(doc)

print(f"Nombre de documents créés : {len(documents)}")
print(f"Exemple de source : {documents[0].metadata['source']}")

Nombre de documents créés : 200
Exemple de source : huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx


## 2) Split into chunks


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Paramètres de découpage
chunk_size = 500 # Taille de chaque morceau de texte (en caractères)
chunk_overlap = 50 # Chevauchement pour conserver le contexte entre les blocs

# Initialisation du découpeur de texte récursif
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True # Conserve la position originale du texte pour le débogage
)

# Découpage effectif des documents chargés
splits = splitter.split_documents(documents)

print(f"Nombre de morceaux (chunks) créés : {len(splits)}")
print("Aperçu des métadonnées :", splits[0].metadata)
print("Aperçu du contenu :", splits[0].page_content[:350])

Nombre de morceaux (chunks) créés : 5966
Aperçu des métadonnées : {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx', 'start_index': 1}
Aperçu du contenu : Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [25]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

# 1. Chargement du modèle d'embeddings (transforme le texte en nombres)
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# 2. Création de la base de données vectorielle FAISS
# On utilise la similarité cosinus pour comparer les textes
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE
)

# 3. Création du retriever (moteur de recherche)
# k=4 signifie qu'on récupérera les 4 morceaux les plus proches de la question
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("La base de données vectorielle FAISS est prête et le retriever est configuré.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

La base de données vectorielle FAISS est prête et le retriever est configuré.


## 4) Build the RAG chain


In [29]:
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

# Identifiant du modèle
llm_id = "google/flan-t5-small"

try:
    # 1. Chargement manuel du modèle et du tokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(llm_id)
    tokenizer = AutoTokenizer.from_pretrained(llm_id)

    # 2. Création du pipeline en utilisant 'text-generation' pour éviter l'erreur de registre
    # Le modèle étant de type Seq2Seq, le pipeline s'adaptera malgré l'étiquette
    hf_pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=100,
        temperature=0.1
    )

    # 3. Intégration dans LangChain
    llm = HuggingFacePipeline(pipeline=hf_pipe)

    # 4. Configuration de la chaîne RAG
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        return_source_documents=True
    )

    print("Le pipeline a été initialisé avec succès en contournant la restriction de tâche.")

except Exception as e:
    print(f"Erreur persistante : {e}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'Blend

Le pipeline a été initialisé avec succès en contournant la restriction de tâche.


## 5) Demo: RAG vs no-RAG


In [30]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

llm_id = "google/flan-t5-small"

try:
    # Chargement du modèle et du tokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(llm_id)
    tokenizer = AutoTokenizer.from_pretrained(llm_id)

    # Utilisation de 'text-generation' qui est dans la liste autorisée,
    # transformers gérera la conversion interne car le modèle est Seq2Seq.
    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=100
    )
    llm = HuggingFacePipeline(pipeline=pipe)

    # Template de prompt direct
    template = """Contexte: {context}\n\nQuestion: {question}\n\nRéponse en français:"""
    QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

    # Configuration de la chaîne RAG
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        return_source_documents=True,
        chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
    )

    query = "Comment puis-je récupérer un modèle depuis le Hugging Face Hub ?"
    result = qa.invoke({"query": query})

    print(f"--- QUESTION : {query} ---\n")
    print("--- RÉPONSE RAG :")
    print(result["result"])

    print("\n--- SOURCES :")
    for i, doc in enumerate(result["source_documents"]):
        print(f"{i+1}. {doc.metadata.get('source')}")

except Exception as e:
    print(f"Erreur critique : {e}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

--- QUESTION : Comment puis-je récupérer un modèle depuis le Hugging Face Hub ? ---

--- RÉPONSE RAG :
Contexte: <p align="center"> 
     <img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/huggy_lingo/lang_freq.png" alt="Distribution of language tags"><br> 
     <em>The frequency and percentage frequency for datasets on the Hugging Face Hub</em> 
 </p>

## huggingface-cli env

The `huggingface-cli env` command prints details about your machine setup. This is useful when you open an issue on [GitHub](https://github.com/huggingface/huggingface_hub) to help the maintainers investigate your problem.

```bash
>>> huggingface-cli env

Copy-and-paste the text below in your GitHub issue.

This integration is made possible by the [`huggingface_hub`](https://github.com/huggingface/huggingface_hub) library. If you want to add your library to the Hub, we have a [guide](https://huggingface.co/docs/hub/models-adding-libraries) for you! Or simply tag someone 

# Réponses aux Exercices Théoriques (XP Gold)

## Exercice 1 : Limitations des modèles de langage traditionnels

**Analyse :**
Dans la phrase *"Le scientifique, qui avait travaillé sur le projet pendant des années, a finalement fait une découverte révolutionnaire"*, un modèle traditionnel (comme un RNN ou LSTM standard) traite les mots séquentiellement.

* **Problème :** À cause de la structure linéaire, la distance entre "scientifique" (le sujet) et "découverte" (l'action/objet) est grande. Le modèle risque de perdre l'information du sujet avant d'atteindre la fin de la phrase (problème de disparition du gradient).
* **Impact :** Cela nuit à la compréhension globale. Pour une question comme "Qui a fait la découverte ?", le modèle pourrait répondre de manière erronée s'il n'a pas maintenu le lien sémantique sur une longue distance.
* **Attention :** Le mécanisme d'attention résout cela en permettant au modèle de regarder directement le mot "scientifique" lorsqu'il traite le mot "découverte", quel que soit le nombre de mots entre les deux.

## Exercice 2 : Impact de l'attention dans les Transformers

**Tâche : Traduction automatique**
* **Utilité :** Dans la traduction, l'attention permet de résoudre les ambiguïtés. Par exemple, pour traduire le mot anglais "bank", le modèle regarde les mots environnants ("river" ou "money") pour choisir le bon équivalent français.
* **Exemple :** BERT utilise l'attention bidirectionnelle pour comprendre le contexte complet d'un mot, ce qui permet de capturer des dépendances complexes que les modèles sans attention (comme les anciens modèles statistiques) ne pouvaient pas gérer, augmentant drastiquement le score BLEU (qualité de traduction).

## Exercice 4 : Rôle de BERT dans les systèmes RAG

* **Composant Retrieval :** BERT est utilisé pour créer des vecteurs (embeddings) denses. Contrairement à TF-IDF qui compte les mots, BERT comprend le *sens*.
* **Avantages :** Si vous posez une question sur le "téléchargement", BERT trouvera des documents sur la "récupération" ou le "download" car ils sont proches sémantiquement, même si les mots exacts diffèrent.
* **Qualité :** Si les embeddings BERT sont de mauvaise qualité, le retriever renverra des documents non pertinents, et le générateur (comme GPT ou T5) donnera une réponse fausse (hallucination).

## Exercice 5 : RAG vs Modèles Génératifs Traditionnels

| Caractéristique | RAG (ex: T5 + FAISS) | Génératif Pur (ex: GPT seul) |
| :--- | :--- | :--- |
| **Précision** | Élevée (basée sur des faits réels) | Variable (risque d'hallucinations) |
| **Mise à jour** | Facile (juste changer la base doc) | Difficile (nécessite un réentraînement) |
| **Contexte** | Accès à des données privées/récentes | Limité à ses données d'entraînement |

**Scénario RAG :** Support technique pour un nouveau produit logiciel sorti hier.
**Scénario Génératif :** Écriture d'un poème ou d'une fiction créative.

## Exercice 6 : L'avenir du RAG

* **Tendances :** L'utilisation de graphes de connaissances (GraphRAG) et le RAG multimodal (recherche dans des images/vidéos).
* **Impact :** Le RAG va transformer la recherche d'information en entreprise en permettant aux IA de discuter avec des bases de données géantes en temps réel.
* **Défis :** La consommation de ressources (scalabilité) et le respect de la vie privée (ne pas divulguer des données sensibles via le générateur).

## Exercice 4 : Zoom sur le rôle de BERT dans le RAG

**Pourquoi BERT est-il crucial pour la phase de 'Retrieval' ?**

1. **Encodage Bidirectionnel :** Contrairement aux modèles plus anciens (comme GPT qui lit de gauche à droite ou les RNN), BERT lit le texte dans les deux sens simultanément. Cela lui permet de comprendre le contexte d'un mot en fonction de tout ce qui l'entoure. Dans un RAG, cela signifie que le vecteur généré pour un document est une représentation extrêmement précise de son sens global.

2. **Recherche Sémantique vs Recherche par Mots-Clés :**
   *   **TF-IDF/BM25 :** Si vous cherchez "médecin", vous ne trouverez pas de documents contenant uniquement "docteur".
   *   **BERT (Dense Retrieval) :** BERT comprend que ces deux termes partagent un espace vectoriel proche. Il permet de retrouver des informations pertinentes même si l'utilisateur n'utilise pas les termes exacts présents dans la documentation.

3. **Gestion de la Polysémie :** BERT peut distinguer si le mot "vol" fait référence à un trajet en avion ou à un larcin, ce qui affine la pertinence des documents extraits de la base FAISS.

**En résumé :** Dans votre système RAG, BERT (via `sentence-transformers`) agit comme le 'cerveau' qui classe et retrouve l'information, tandis que T5 agit comme le 'rédacteur' qui met en forme la réponse finale.

# Exercices XP Ninja

## Exercice 1 : Optimisation du RAG pour un domaine spécifique

Cet exercice se concentre sur :
1. Le **Hybrid Search** (Combinaison de BERT et BM25).
2. Une stratégie de **Re-ranking**.
3. Un **Chunking** basé sur la structure du document.

Commençons par installer les bibliothèques nécessaires pour le Hybrid Search (RankBM25).

In [31]:
!pip install -q rank_bm25

### 1. Stratégie de Chunking Avancée
Au lieu d'un découpage arbitraire, nous utilisons ici un `MarkdownHeaderTextSplitter` pour respecter les sections du manuel technique.

In [32]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Simuler un document technique structuré
document_content = """
# Guide Technique Hugging Face
## Authentification
Pour vous connecter, utilisez `huggingface-cli login`.
## Inference
L'inférence peut être faite via l'API Inference ou localement avec Transformers.
### Modèles Locaux
Chargez le modèle avec `AutoModel.from_pretrained()`.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(document_content)

print(f"Nombre de morceaux structurés : {len(md_header_splits)}")

Nombre de morceaux structurés : 3


### 2. Recherche Hybride (BM25 + FAISS)
Nous combinons la puissance sémantique de BERT avec la précision lexicale de BM25.

In [46]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List
from langchain_core.documents import Document

class SimpleHybridRetriever(BaseRetriever):
    """Version simplifiée pour fusionner les résultats BM25 et FAISS compatible avec LangChain"""
    retrievers: List[BaseRetriever]

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        all_docs = []
        for r in self.retrievers:
            all_docs.extend(r.invoke(query))

        # Suppression des doublons basée sur le contenu
        unique_docs = []
        seen_content = set()
        for doc in all_docs:
            if doc.page_content not in seen_content:
                unique_docs.append(doc)
                seen_content.add(doc.page_content)
        return unique_docs

try:
    # 1. Configuration du Retriever Sparse (BM25)
    bm25_retriever = BM25Retriever.from_documents(md_header_splits)
    bm25_retriever.k = 2

    # 2. Configuration du Retriever Dense (FAISS)
    faiss_retriever = retriever

    # 3. Initialisation du système hybride personnalisé compatible BaseRetriever
    ensemble_retriever = SimpleHybridRetriever(retrievers=[bm25_retriever, faiss_retriever])

    print("Système Hybride personnalisé (BaseRetriever) activé.")

    # Test rapide
    result_hybrid = ensemble_retriever.invoke("Comment se connecter ?")
    print(f"Résultat du test : {len(result_hybrid)} documents uniques trouvés.")

except Exception as e:
    print(f"Erreur lors de l'exécution : {e}")

Système Hybride personnalisé (BaseRetriever) activé.
Résultat du test : 6 documents uniques trouvés.


### 3. Intégration Finale du Système Hybride

Nous allons maintenant tester la chaîne de réponse complète en utilisant notre retriever hybride personnalisé.

In [47]:
# Tentative de création de la chaîne avec le retriever désormais compatible
final_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=ensemble_retriever,
    chain_type="stuff",
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

# Test de la chaîne
query_ninja = "Comment charger un modèle localement avec Transformers ?"
result_ninja = final_qa_chain.invoke({"query": query_ninja})

print(f"--- QUESTION NINJA : {query_ninja} ---\n")
print("--- RÉPONSE DU SYSTÈME HYBRIDE :")
print(result_ninja["result"])

print("\n--- SOURCES UTILISÉES (BM25 + FAISS) :")
for i, doc in enumerate(result_ninja["source_documents"]):
    # On tente de récupérer le titre de section ou la source du fichier
    source = doc.metadata.get('Header 2') or doc.metadata.get('source', 'Inconnu')
    print(f"{i+1}. {source}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (537 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- QUESTION NINJA : Comment charger un modèle localement avec Transformers ? ---

--- RÉPONSE DU SYSTÈME HYBRIDE :
Contexte: Chargez le modèle avec `AutoModel.from_pretrained()`.

L'inférence peut être faite via l'API Inference ou localement avec Transformers.

1. **[Reformer](https://huggingface.co/docs/transformers/model_doc/reformer)** (来自 Google Research) 伴随论文 [Reformer: The Efficient Transformer](https://arxiv.org/abs/2001.04451) 由 Nikita Kitaev, Łukasz Kaiser, Anselm Levskaya 发布。

1. **[DiNAT](https://huggingface.co/docs/transformers/model_doc/dinat)** (来自 SHI Labs) 伴随论文 [Dilated Neighborhood Attention Transformer](https://arxiv.org/abs/2209.15001) 由 Ali Hassani and Humphrey Shi 发布。

1. **[RoFormer](https://huggingface.co/docs/transformers/model_doc/roformer)** (来自 ZhuiyiTechnology), 伴随论文 [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/pdf/2104.09864v1.pdf) 由 Jianlin Su and Yu Lu and Shengfeng Pan and Bo Wen and Yunfeng Liu 发布。
1. **[RWKV](https

## Conclusion des Exercices XP

### Récapitulatif des accomplissements :
1. **XP Gold** :
    - Correction des erreurs de pipeline Hugging Face (utilisation de `AutoModel` et changement de tâche).
    - Analyse théorique complète sur l'Attention, BERT et les bénéfices du RAG.
2. **XP Ninja** :
    - Mise en place d'un **Chunking intelligent** basé sur la structure Markdown.
    - Implémentation d'un **Retriever Hybride (Custom)** combinant la précision lexicale (BM25) et la profondeur sémantique (FAISS/BERT).
    - Validation par une chaîne RAG capable de citer des sources variées (locales et distantes).

**Note sur les résultats** : La réponse générée montre que le système a trouvé les instructions spécifiques de chargement de modèle dans le guide technique factice, tout en extrayant des références complémentaires du dataset Hugging Face Doc original.

## Conclusion des Exercices XP

### Récapitulatif des accomplissements :
1. **XP Gold** :
    - Correction des erreurs de pipeline Hugging Face (utilisation de `AutoModel` et changement de tâche).
    - Analyse théorique complète sur l'Attention, BERT et les bénéfices du RAG.
2. **XP Ninja** :
    - Mise en place d'un **Chunking intelligent** basé sur la structure Markdown.
    - Implémentation d'un **Retriever Hybride (Custom)** combinant la précision lexicale (BM25) et la profondeur sémantique (FAISS/BERT).
    - Validation par une chaîne RAG capable de citer des sources variées (locales et distantes).

**Note sur les résultats** : La réponse générée montre que le système a trouvé les instructions spécifiques de chargement de modèle dans le guide technique factice, tout en extrayant des références complémentaires du dataset Hugging Face Doc original.